In [1]:
import os
import random
import shutil

# Perform train test split
TRAIN_RATIO = 0.8
VALIDATION_RATIO = 0.1
TEST_RATIO = 0.1
INPUT_FOLDER = 'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/archive (3)/Garbage classification/Garbage classification'
OUTPUT_FOLDER = 'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification'
CLASSES = ['cardboard', 'glass', 'metal', 'paper', 'plastic', 'trash']

for class_name in CLASSES:
    # Get the filename of all images in that class
    images = [
        f
        for f in os.listdir(INPUT_FOLDER + '/' + class_name)
    ]
    
    # Shuffle images
    random.shuffle(images)

    # Calculate split indexes
    train_end = int(len(images) * TRAIN_RATIO)
    validation_end = train_end + int(len(images) * VALIDATION_RATIO)

    # Copy files to respective folders
    for i, img in enumerate(images):
        src = INPUT_FOLDER + '/' + class_name + '/' + img
        if i < train_end:
            dst = OUTPUT_FOLDER + '/train/' + class_name + '/' + img
        elif i < validation_end:
            dst = OUTPUT_FOLDER + '/validation/' + class_name + '/' + img
        else:
            dst = OUTPUT_FOLDER + '/test/' + class_name + '/' + img
        shutil.copy(src, dst)


In [4]:
import pathlib

# Folder of train dataset, validation dataset and test dataset
train_dataset_directory = pathlib.Path('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/train')
validation_dataset_directory = pathlib.Path('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/validation')
test_dataset_directory = pathlib.Path('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/test')

In [5]:
# Print size of train dataset, validation dataset and test dataset
print('Train dataset count:', len(list(train_dataset_directory.glob('*/*'))))
print('Validation dataset count:', len(list(validation_dataset_directory.glob('*/*'))))
print('Test dataset count:', len(list(test_dataset_directory.glob('*/*'))))

Train dataset count: 2019
Validation dataset count: 251
Test dataset count: 257


In [15]:
import torch

# Check GPU availability
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")
if device.type == "cuda":
    print(f"NVIDIA GPU Detected: {torch.cuda.get_device_name(0)}")

Using device: cuda
NVIDIA GPU Detected: NVIDIA GeForce RTX 4070 Laptop GPU


In [1]:
import sys
import torch

print("Python Executable Path:", sys.executable)
print("PyTorch Version      :", torch.__version__)
print("CUDA Available?      :", torch.cuda.is_available())

Python Executable Path: c:\Users\user\AppData\Local\Programs\Python\Python312\python.exe
PyTorch Version      : 2.5.1+cu121
CUDA Available?      : True


In [2]:
NUM_CLASSES = 6
INPUT_SHAPE = (3, 224, 224)       # (Channels, Height, Width)
BATCH_SIZE = 32
PHASE_1_EPOCHS = 5
PHASE_2_EPOCHS = 10
OPTUNA_TRIALS = 12                # Total hyperparameter combinations to evaluate
MobileNetV3_Small_ONNX_EXPORT_PATH = "C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small.onnx"

In [3]:
from torchvision import transforms

train_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(15),  # Rotate 15 degrees
    transforms.ColorJitter(contrast=0.2),   # Adjust the contrast between 0.8 and 1.2 (+-0.2)
    transforms.ToTensor(),  # Convert PIL image (0-255) to Tensor image (0-1)
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])     # Mean and standard deviation for ImageNet images
])

val_transform = transforms.Compose([
    transforms.Resize((224,224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

In [4]:
from torchvision import datasets
from torch.utils.data import DataLoader

# Function for returning custom data loader
def get_dataloaders():
    train_dataset = datasets.ImageFolder('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/train', transform=train_transform)
    val_dataset = datasets.ImageFolder('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/validation', transform=val_transform)
    
    train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
    val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False)
    return train_loader, val_loader

In [14]:
from torchvision import models
import torch.nn as nn

# Function to return MobileNetV3-Small model with classification head
def build_mobilenet_v3(num_classes=NUM_CLASSES):
    model = models.mobilenet_v3_small(pretrained=True)
    
    # Replace final Linear classification layer
    in_features = model.classifier[3].in_features
    model.classifier[3] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

In [5]:
# Function for training (one epoch)
def train_one_epoch(model, dataloader, criterion, optimizer):
    model.train()
    running_loss, correct = 0.0, 0
    for inputs, labels in dataloader:
        inputs, labels = inputs.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, labels)
        loss.backward()
        optimizer.step()
        running_loss += loss.item() * inputs.size(0)
        correct += (outputs.argmax(1) == labels).sum().item()
    return running_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

In [6]:
# Function for validation
def validate(model, dataloader, criterion):
    model.eval()
    running_loss, correct = 0.0, 0
    with torch.no_grad():
        for inputs, labels in dataloader:
            inputs, labels = inputs.to(device), labels.to(device)
            outputs = model(inputs)
            loss = criterion(outputs, labels)
            running_loss += loss.item() * inputs.size(0)
            correct += (outputs.argmax(1) == labels).sum().item()
    return running_loss / len(dataloader.dataset), correct / len(dataloader.dataset)

In [7]:
import copy
import torch.optim as optim
import optuna

# Use optuna to tune hyperparameters using Bayesian optimisation
class OptunaObjective:
    def __init__(self, warm_state_dict, train_loader, val_loader, build_backbone_function):
        self.warm_state_dict = warm_state_dict
        self.train_loader = train_loader
        self.val_loader = val_loader
        self.build_backbone_function = build_backbone_function

    def __call__(self, trial):
        # Define Hyperparameter Search Space
        unfreeze_pct = trial.suggest_categorical("unfreeze_pct", [0.10, 0.20, 0.30, 0.40])
        lr = trial.suggest_float("learning_rate", 1e-5, 1e-4, log=True)
        optimizer_name = trial.suggest_categorical("optimizer", ["adamw", "sgd"])

        # Instantiate fresh model & reload Phase 1 warm weights
        model = self.build_backbone_function().to(device)
        model.load_state_dict(copy.deepcopy(self.warm_state_dict))

        # Freeze all layers initially
        for param in model.features.parameters():
            param.requires_grad = False

        # Dynamically unfreeze top N percentage of feature blocks
        total_blocks = len(list(model.features.children()))
        num_unfreeze_blocks = int(total_blocks * unfreeze_pct)
        
        for block in list(model.features.children())[-num_unfreeze_blocks:]:
            for param in block.parameters():
                param.requires_grad = True

        # Configure Optimizer
        trainable_params = [p for p in model.parameters() if p.requires_grad]
        if optimizer_name == "adamw":
            optimizer = optim.AdamW(trainable_params, lr=lr, weight_decay=1e-4)
        else:
            optimizer = optim.SGD(trainable_params, lr=lr, momentum=0.9)

        criterion = nn.CrossEntropyLoss()

        # Train Trial Model
        best_val_acc = 0.0
        for epoch in range(PHASE_2_EPOCHS):
            train_loss, train_acc = train_one_epoch(model, self.train_loader, criterion, optimizer)
            val_loss, val_acc = validate(model, self.val_loader, criterion)
            
            if val_acc > best_val_acc:
                best_val_acc = val_acc

            # Optuna Pruning (early stops unpromising trials)
            trial.report(val_acc, epoch)
            if trial.should_prune():
                raise optuna.exceptions.TrialPruned()

        # Delete model
        del model

        return best_val_acc

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [8]:
# Get loader for train and validation datasets
train_loader, val_loader = get_dataloaders()

In [9]:
import random
import numpy as np

# Set seed
random.seed(0)
np.random.seed(0)
torch.backends.cudnn.deterministic = True    
torch.backends.cudnn.benchmark = False

In [19]:
# Fine tune MobileNetV3-Small model
model = build_mobilenet_v3().to(device)
    
# Freeze feature backbone
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

for epoch in range(PHASE_1_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)
    print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Save warm weights in memory for Optuna trials
warm_state_dict = copy.deepcopy(model.state_dict())

# Remove model
del model

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Phase 1 - Epoch 1/5 | Train Acc: 0.6419 | Val Acc: 0.7331
Phase 1 - Epoch 2/5 | Train Acc: 0.7464 | Val Acc: 0.7530
Phase 1 - Epoch 3/5 | Train Acc: 0.7796 | Val Acc: 0.7610
Phase 1 - Epoch 4/5 | Train Acc: 0.7900 | Val Acc: 0.7570
Phase 1 - Epoch 5/5 | Train Acc: 0.8202 | Val Acc: 0.7450


In [20]:
# Instantiate Optuna Study (Hyperparameter tuning using Bayesian optimisation)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
objective_fn = OptunaObjective(warm_state_dict, train_loader, val_loader, build_mobilenet_v3)
    
study.optimize(objective_fn, n_trials=OPTUNA_TRIALS)

# Output Best Results
print(f"Best Validation Accuracy: {study.best_value:.4f}")
print("Optimal Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

C:\Users\user\AppData\Local\Temp\ipykernel_16020\2097342654.py:2: FutureWarning: __init__() got {'consider_prior'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'consider_prior', 'prior_weight', 'consider_magic_clip', 'consider_endpoints', 'n_startup_trials', 'n_ei_candidates', 'gamma', 'weights', 'seed'] in __init__() have been deprecated since v4.4.0. They will be replaced with the corresponding keyword arguments in v6.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v4.4.0 for details.
  study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\samplers\_tpe\sampler.py:82: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  optuna_warn(msg, FutureW

Best Validation Accuracy: 0.8685
Optimal Hyperparameters:
  - unfreeze_pct: 0.3
  - learning_rate: 6.612865475986788e-05
  - optimizer: adamw


In [21]:
# Re-train model with best hyperparameters
best_params = study.best_params
final_model = build_mobilenet_v3().to(device)
final_model.load_state_dict(warm_state_dict)

# Apply best unfreeze depth
for param in final_model.features.parameters():
    param.requires_grad = False
        
total_blocks = len(list(final_model.features.children()))
num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
for block in list(final_model.features.children())[-num_unfreeze_blocks:]:
    for param in block.parameters():
        param.requires_grad = True

# Apply best optimizer
trainable_params = [p for p in final_model.parameters() if p.requires_grad]
if best_params["optimizer"] == "adamw":
    final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
else:
    final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

# Train final model
for epoch in range(PHASE_2_EPOCHS):
    train_loss, train_acc = train_one_epoch(final_model, train_loader, criterion, final_opt)
    val_loss, val_acc = validate(final_model, val_loader, criterion)
    print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

Phase 2 - Epoch 6/15 | Train Acc: 0.8638 | Val Acc: 0.8127
Phase 2 - Epoch 7/15 | Train Acc: 0.8707 | Val Acc: 0.8247
Phase 2 - Epoch 8/15 | Train Acc: 0.8821 | Val Acc: 0.8207
Phase 2 - Epoch 9/15 | Train Acc: 0.8965 | Val Acc: 0.8287
Phase 2 - Epoch 10/15 | Train Acc: 0.9029 | Val Acc: 0.8207
Phase 2 - Epoch 11/15 | Train Acc: 0.9019 | Val Acc: 0.8327
Phase 2 - Epoch 12/15 | Train Acc: 0.9153 | Val Acc: 0.8446
Phase 2 - Epoch 13/15 | Train Acc: 0.9153 | Val Acc: 0.8367
Phase 2 - Epoch 14/15 | Train Acc: 0.9178 | Val Acc: 0.8327
Phase 2 - Epoch 15/15 | Train Acc: 0.9351 | Val Acc: 0.8526


In [ ]:
import os

# Export model to Open Neural Network Exchange (ONNX) to be run outside PyTorch
# Ensure export directory exists
os.makedirs(os.path.dirname(MobileNetV3_Small_ONNX_EXPORT_PATH), exist_ok=True)

# Export to ONNX
final_model.eval()  # Set model in evaluation mode
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    final_model,
    dummy_input,
    MobileNetV3_Small_ONNX_EXPORT_PATH,
    export_params=True,
    opset_version=14,   # Version of ONNX
    do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
    input_names=["input"],  # Key to be referred when giving input during inference
    output_names=["output"],    # Key to be referred when obtaining output during inference
    dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

# Delete model
del final_model

In [25]:
import onnxruntime as ort
import numpy as np
import time

# Test model
BATCH_SIZE = 1  # Use BATCH_SIZE = 1 for real-time latency (ms per image)

# Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
# Load session with CUDA acceleration
session = ort.InferenceSession(
    MobileNetV3_Small_ONNX_EXPORT_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"Active Provider  : {session.get_providers()[0]}")

# Transform data
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/test', transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False)

# GPU warmup (so that latency is accurate)
dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

for _ in range(20):
    session.run([output_name], {input_name: dummy_warmup_input})

# Synchronize GPU after warmup
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Test accuracy and latency
total_samples = 0
correct_predictions = 0
latency_list = []

for images, labels in test_loader:
    # Convert PyTorch tensor to NumPy array (Float32)
    input_numpy = images.numpy()
    
    # Precise GPU Timing Block
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Clear pending GPU tasks
        
    start_time = time.perf_counter()
    
    # Run Inference
    outputs = session.run([output_name], {input_name: input_numpy})
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Wait for GPU to finish computing
        
    end_time = time.perf_counter()
    
    # Calculate Latency in milliseconds
    batch_latency_ms = (end_time - start_time) * 1000
    latency_list.append(batch_latency_ms)
    
    # Accuracy Calculation
    logits = outputs[0]
    preds = np.argmax(logits, axis=1)
    correct_predictions += np.sum(preds == labels.numpy())
    total_samples += images.size(0)

# Display results
avg_accuracy = (correct_predictions / total_samples)
avg_latency = np.mean(latency_list)

print(f"Test Accuracy: {avg_accuracy:.4f}")
print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8755
Average Inference Latency: 2.278 ms / sample


In [26]:
import pandas as pd

# Run remaining 4 seeds for MobileNetV3-Small
test_accuracies = []
latencies_ms = []
SEEDS = [0, 198, 473, 5784, 96312]

test_accuracies.append(avg_accuracy)
latencies_ms.append(avg_latency)

for seed in SEEDS:
    # Skip the first seed
    if seed == 0:
        continue

    # Set seed
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True    
    torch.backends.cudnn.benchmark = False
    print(f'\nSeed: {seed}')

    BATCH_SIZE = 32

    # Fine tune MobileNetV3-Small model
    # Phase 1
    model = build_mobilenet_v3().to(device)
    
    # Freeze feature backbone
    for param in model.features.parameters():
        param.requires_grad = False

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

    for epoch in range(PHASE_1_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Phase 2
    # Apply best unfreeze depth
    for param in model.features.parameters():
        param.requires_grad = False
        
    total_blocks = len(list(model.features.children()))
    num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
    for block in list(model.features.children())[-num_unfreeze_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    # Apply best optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if best_params["optimizer"] == "adamw":
        final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
    else:
        final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

    # Train final model
    for epoch in range(PHASE_2_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, final_opt)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    MobileNetV3_Small_ONNX_EXPORT_PATH_adjust_with_seed = f'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-{seed}.onnx'

    # Export to ONNX
    model.eval()  # Set model in evaluation mode
    dummy_input = torch.randn(1, 3, 224, 224, device=device)

    torch.onnx.export(
        model,
        dummy_input,
        MobileNetV3_Small_ONNX_EXPORT_PATH_adjust_with_seed,
        export_params=True,
        opset_version=14,   # Version of ONNX
        do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
        input_names=["input"],  # Key to be referred when giving input during inference
        output_names=["output"],    # Key to be referred when obtaining output during inference
        dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
            "input": {0: "batch_size"},
            "output": {0: "batch_size"}
        }
    )

    # Delete model
    del model

    BATCH_SIZE = 1

    # Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
    # Load session with CUDA acceleration
    session = ort.InferenceSession(
        MobileNetV3_Small_ONNX_EXPORT_PATH_adjust_with_seed,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    print(f"Active Provider  : {session.get_providers()[0]}")

    # GPU warmup (so that latency is accurate)
    dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

    for _ in range(20):
        session.run([output_name], {input_name: dummy_warmup_input})

    # Synchronize GPU after warmup
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Test accuracy and latency
    total_samples = 0
    correct_predictions = 0
    latency_list = []

    for images, labels in test_loader:
        # Convert PyTorch tensor to NumPy array (Float32)
        input_numpy = images.numpy()
    
        # Precise GPU Timing Block
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Clear pending GPU tasks
        
        start_time = time.perf_counter()
    
        # Run Inference
        outputs = session.run([output_name], {input_name: input_numpy})
    
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Wait for GPU to finish computing
        
        end_time = time.perf_counter()
    
        # Calculate Latency in milliseconds
        batch_latency_ms = (end_time - start_time) * 1000
        latency_list.append(batch_latency_ms)
    
        # Accuracy Calculation
        logits = outputs[0]
        preds = np.argmax(logits, axis=1)
        correct_predictions += np.sum(preds == labels.numpy())
        total_samples += images.size(0)

    # Display results
    avg_accuracy = (correct_predictions / total_samples)
    avg_latency = np.mean(latency_list)

    print(f"Test Accuracy: {avg_accuracy:.4f}")
    print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

    test_accuracies.append(avg_accuracy)
    latencies_ms.append(avg_latency)


Seed: 198


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=MobileNet_V3_Small_Weights.IMAGENET1K_V1`. You can also use `weights=MobileNet_V3_Small_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Phase 1 - Epoch 1/5 | Train Acc: 0.6166 | Val Acc: 0.7291
Phase 1 - Epoch 2/5 | Train Acc: 0.7296 | Val Acc: 0.7211
Phase 1 - Epoch 3/5 | Train Acc: 0.7746 | Val Acc: 0.7769
Phase 1 - Epoch 4/5 | Train Acc: 0.7786 | Val Acc: 0.7729
Phase 1 - Epoch 5/5 | Train Acc: 0.8053 | Val Acc: 0.7729
Phase 2 - Epoch 6/15 | Train Acc: 0.8479 | Val Acc: 0.7928
Phase 2 - Epoch 7/15 | Train Acc: 0.8722 | Val Acc: 0.8008
Phase 2 - Epoch 8/15 | Train Acc: 0.8846 | Val Acc: 0.8207
Phase 2 - Epoch 9/15 | Train Acc: 0.8866 | Val Acc: 0.8247
Phase 2 - Epoch 10/15 | Train Acc: 0.8940 | Val Acc: 0.8247
Phase 2 - Epoch 11/15 | Train Acc: 0.9074 | Val Acc: 0.8406
Phase 2 - Epoch 12/15 | Train Acc: 0.9143 | Val Acc: 0.8486
Phase 2 - Epoch 13/15 | Train Acc: 0.9138 | Val Acc: 0.8247
Phase 2 - Epoch 14/15 | Train Acc: 0.9222 | Val Acc: 0.8406
Phase 2 - Epoch 15/15 | Train Acc: 0.9257 | Val Acc: 0.8406


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8482
Average Inference Latency: 2.253 ms / sample

Seed: 473
Phase 1 - Epoch 1/5 | Train Acc: 0.6147 | Val Acc: 0.6773
Phase 1 - Epoch 2/5 | Train Acc: 0.7390 | Val Acc: 0.7171
Phase 1 - Epoch 3/5 | Train Acc: 0.7682 | Val Acc: 0.7928
Phase 1 - Epoch 4/5 | Train Acc: 0.7855 | Val Acc: 0.7928
Phase 1 - Epoch 5/5 | Train Acc: 0.8158 | Val Acc: 0.7888
Phase 2 - Epoch 6/15 | Train Acc: 0.8470 | Val Acc: 0.8088
Phase 2 - Epoch 7/15 | Train Acc: 0.8653 | Val Acc: 0.8048
Phase 2 - Epoch 8/15 | Train Acc: 0.8791 | Val Acc: 0.8367
Phase 2 - Epoch 9/15 | Train Acc: 0.8871 | Val Acc: 0.8486
Phase 2 - Epoch 10/15 | Train Acc: 0.8920 | Val Acc: 0.8287
Phase 2 - Epoch 11/15 | Train Acc: 0.9069 | Val Acc: 0.8446
Phase 2 - Epoch 12/15 | Train Acc: 0.9054 | Val Acc: 0.8526
Phase 2 - Epoch 13/15 | Train Acc: 0.9123 | Val Acc: 0.8566
Phase 2 - Epoch 14/15 | Train Acc: 0.9099 | Val Acc: 0.8645
Phase 2 - Epoch 15/15 | Train Acc: 0.9222 | Val Acc: 0.8

In [27]:
# Record results
MobileNetV3_small_results = pd.DataFrame({
    'seed': SEEDS,
    'test accuracy': test_accuracies,
    'latency in millisecond': latencies_ms
})

MobileNetV3_small_results.to_csv('MobileNetV3-Small-results.csv', index = False)

In [32]:
from onnxruntime.quantization import quantize_static, CalibrationDataReader, QuantType

# Quantize the MobileNetV3-Small model (static quantization)
# quantization converts the 32 point float to 8 bit integer, which saves space
# static quantization calculates the zero crossing and the scale of the inputs and uses it during inference
# so static quantization needs calibration data (part of validation dataset) to get the actual range of input data
# the zero crossing and scale is used to calculate the quantized integer for activation function
# static quantization is more accurate than dynamic quantization, which does not calculate the zero crossing and scale
# Define Data Reader to feed calibration samples
class ImageNetDataReader(CalibrationDataReader):
    def __init__(self, dataset, input_name="input", max_samples=200):
        data_list = []
        # Limit to max_samples to keep calibration fast
        for i in range(min(len(dataset), max_samples)):
            img_tensor, label = dataset[i]
            # Convert PyTorch tensor to NumPy array and add batch dimension (1, C, H, W)
            img_numpy = img_tensor.unsqueeze(0).cpu().numpy()
            data_list.append({input_name: img_numpy})

        self.enum_data = iter(data_list)

    def get_next(self):
        return next(self.enum_data, None)

# Create sample calibration data
calibration_data = datasets.ImageFolder('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/validation', transform=val_transform)
data_reader = ImageNetDataReader(calibration_data)

# Perform Static Quantization
quantize_static(
    model_input="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small.onnx",
    model_output="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Quantized.onnx",
    calibration_data_reader=data_reader,
    quant_format=QuantType.QInt8
)

data_reader = ImageNetDataReader(calibration_data)
quantize_static(
    model_input="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-198.onnx",
    model_output="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-198-Quantized.onnx",
    calibration_data_reader=data_reader,
    quant_format=QuantType.QInt8
)

data_reader = ImageNetDataReader(calibration_data)
quantize_static(
    model_input="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-473.onnx",
    model_output="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-473-Quantized.onnx",
    calibration_data_reader=data_reader,
    quant_format=QuantType.QInt8
)

data_reader = ImageNetDataReader(calibration_data)
quantize_static(
    model_input="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-5784.onnx",
    model_output="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-5784-Quantized.onnx",
    calibration_data_reader=data_reader,
    quant_format=QuantType.QInt8
)

data_reader = ImageNetDataReader(calibration_data)
quantize_static(
    model_input="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-96312.onnx",
    model_output="C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-96312-Quantized.onnx",
    calibration_data_reader=data_reader,
    quant_format=QuantType.QInt8
)

In [ ]:
# Test quantized models
quantized_model_paths = [
    'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Quantized.onnx',
    'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-198-Quantized.onnx',
    'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-473-Quantized.onnx',
    'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-5784-Quantized.onnx',
    'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV3-Small-Seed-96312-Quantized.onnx'
]

test_accuracies = []
latencies_ms = []

for quantized_model_path in quantized_model_paths:
    BATCH_SIZE = 1

    # Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
    # Load session with CUDA acceleration
    session = ort.InferenceSession(
        quantized_model_path,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    print(f"Active Provider  : {session.get_providers()[0]}")

    # GPU warmup (so that latency is accurate)
    dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

    for _ in range(20):
        session.run([output_name], {input_name: dummy_warmup_input})

    # Synchronize GPU after warmup
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Test accuracy and latency
    total_samples = 0
    correct_predictions = 0
    latency_list = []

    for images, labels in test_loader:
        # Convert PyTorch tensor to NumPy array (Float32)
        input_numpy = images.numpy()
    
        # Precise GPU Timing Block
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Clear pending GPU tasks
        
        start_time = time.perf_counter()
    
        # Run Inference
        outputs = session.run([output_name], {input_name: input_numpy})
    
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Wait for GPU to finish computing
        
        end_time = time.perf_counter()
    
        # Calculate Latency in milliseconds
        batch_latency_ms = (end_time - start_time) * 1000
        latency_list.append(batch_latency_ms)
    
        # Accuracy Calculation
        logits = outputs[0]
        preds = np.argmax(logits, axis=1)
        correct_predictions += np.sum(preds == labels.numpy())
        total_samples += images.size(0)

    # Display results
    avg_accuracy = (correct_predictions / total_samples)
    avg_latency = np.mean(latency_list)

    print(f"Test Accuracy: {avg_accuracy:.4f}")
    print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

    test_accuracies.append(avg_accuracy)
    latencies_ms.append(avg_latency)

quantized_results_dataframe = pd.DataFrame({
    'test accuracy (quantized)': test_accuracies,
    'latency in millisecond (quantized)': latencies_ms
})

non_quantized_results_dataframe = pd.read_csv('MobileNetV3-Small-results.csv')

MobileNetV3_small_results = pd.concat([non_quantized_results_dataframe, quantized_results_dataframe], axis=1)
MobileNetV3_small_results.to_csv('MobileNetV3-Small-results.csv', index=False)


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Active Provider  : CPUExecutionProvider
Test Accuracy: 0.2296
Average Inference Latency: 4.411 ms / sample
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.2062
Average Inference Latency: 4.640 ms / sample
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.2568
Average Inference Latency: 4.172 ms / sample
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.2179
Average Inference Latency: 4.259 ms / sample
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.2296
Average Inference Latency: 4.365 ms / sample


In [10]:
import os
import time
import numpy as np
import pandas as pd
import torch
import onnxruntime as ort
from onnxruntime.quantization import quantize_dynamic, QuantType

# Perform dynamic quantization
base_dir = "C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/"

fp32_models = [
    "MobileNetV3-Small.onnx",
    "MobileNetV3-Small-Seed-198.onnx",
    "MobileNetV3-Small-Seed-473.onnx",
    "MobileNetV3-Small-Seed-5784.onnx",
    "MobileNetV3-Small-Seed-96312.onnx"
]

# Quantize all models using Dynamic Quantization
for model_name in fp32_models:
    in_path = os.path.join(base_dir, model_name)
    out_name = model_name.replace(".onnx", "-Dynamic-Quantized.onnx")
    out_path = os.path.join(base_dir, out_name)
    
    quantize_dynamic(
        model_input=in_path,
        model_output=out_path,
        weight_type=QuantType.QUInt8
    )

dynamic_quant_paths = [
    os.path.join(base_dir, m.replace(".onnx", "-Dynamic-Quantized.onnx"))
    for m in fp32_models
]

test_accuracies = []
latencies_ms = []

# Transform data
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

test_dataset = datasets.ImageFolder('C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/test', transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=1, shuffle=False)

# Evaluate Accuracy and Latency
for model_path in dynamic_quant_paths:
    session = ort.InferenceSession(
        model_path,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    # Warmup
    dummy_input = np.random.randn(1, 3, 224, 224).astype(np.float32)
    for _ in range(20):
        session.run([output_name], {input_name: dummy_input})

    if torch.cuda.is_available():
        torch.cuda.synchronize()

    correct_predictions = 0
    total_samples = 0
    latency_list = []

    for images, labels in test_loader:
        input_numpy = images.cpu().numpy().astype(np.float32)

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        start_time = time.perf_counter()
        outputs = session.run([output_name], {input_name: input_numpy})

        if torch.cuda.is_available():
            torch.cuda.synchronize()

        end_time = time.perf_counter()

        latency_list.append((end_time - start_time) * 1000.0)

        preds = np.argmax(outputs[0], axis=1)
        correct_predictions += np.sum(preds == labels.numpy())
        total_samples += images.size(0)

    avg_acc = correct_predictions / total_samples
    avg_lat = np.mean(latency_list)

    print(f"Model: {os.path.basename(model_path)}")
    print(f"  Accuracy: {avg_acc:.4f} | Latency: {avg_lat:.3f} ms/sample")

    test_accuracies.append(avg_acc)
    latencies_ms.append(avg_lat)

# Append to CSV clean and formatted
dynamic_df = pd.DataFrame({
    'Dynamic INT8 Accuracy': test_accuracies,
    'Dynamic INT8 Latency (ms)': latencies_ms
})

existing_results = pd.read_csv('MobileNetV3-Small-results.csv').reset_index(drop=True)
updated_results = pd.concat([existing_results, dynamic_df.reset_index(drop=True)], axis=1)
updated_results.to_csv('MobileNetV3-Small-results.csv', index=False)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Model: MobileNetV3-Small-Dynamic-Quantized.onnx
  Accuracy: 0.2802 | Latency: 12.368 ms/sample
Model: MobileNetV3-Small-Seed-198-Dynamic-Quantized.onnx
  Accuracy: 0.2724 | Latency: 11.922 ms/sample
Model: MobileNetV3-Small-Seed-473-Dynamic-Quantized.onnx
  Accuracy: 0.2607 | Latency: 12.233 ms/sample
Model: MobileNetV3-Small-Seed-5784-Dynamic-Quantized.onnx
  Accuracy: 0.2840 | Latency: 12.084 ms/sample
Model: MobileNetV3-Small-Seed-96312-Dynamic-Quantized.onnx
  Accuracy: 0.2724 | Latency: 12.159 ms/sample


In [17]:
# Function to return MobileNetV2 model with classification head
def build_mobilenet_v2(num_classes=NUM_CLASSES):
    model = models.mobilenet_v2(pretrained=True)
    
    # Replace final Linear classification layer
    in_features = model.classifier[1].in_features
    model.classifier[1] = nn.Sequential(
        nn.Dropout(0.3),
        nn.Linear(in_features, num_classes)
    )
    return model

In [12]:
# Set seed
random.seed(0)
np.random.seed(0)
torch.backends.cudnn.deterministic = True    
torch.backends.cudnn.benchmark = False

# Set batch size
BATCH_SIZE = 32

# Set save model path
MobileNetV2_ONNX_EXPORT_PATH = 'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV2.onnx'

In [18]:
# Fine tune MobileNetV2 model
model = build_mobilenet_v2().to(device)
    
# Freeze feature backbone
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

for epoch in range(PHASE_1_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)
    print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Save warm weights in memory for Optuna trials
warm_state_dict = copy.deepcopy(model.state_dict())

# Remove model
del model

Phase 1 - Epoch 1/5 | Train Acc: 0.5230 | Val Acc: 0.6534
Phase 1 - Epoch 2/5 | Train Acc: 0.7023 | Val Acc: 0.7092
Phase 1 - Epoch 3/5 | Train Acc: 0.7370 | Val Acc: 0.7371
Phase 1 - Epoch 4/5 | Train Acc: 0.7504 | Val Acc: 0.7649
Phase 1 - Epoch 5/5 | Train Acc: 0.7637 | Val Acc: 0.7729


In [20]:
# Instantiate Optuna Study (Hyperparameter tuning using Bayesian optimisation)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
objective_fn = OptunaObjective(warm_state_dict, train_loader, val_loader, build_mobilenet_v2)
    
study.optimize(objective_fn, n_trials=OPTUNA_TRIALS)

# Output Best Results
print(f"Best Validation Accuracy: {study.best_value:.4f}")
print("Optimal Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

C:\Users\user\AppData\Local\Temp\ipykernel_28924\2379030106.py:2: FutureWarning: __init__() got {'consider_prior'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'consider_prior', 'prior_weight', 'consider_magic_clip', 'consider_endpoints', 'n_startup_trials', 'n_ei_candidates', 'gamma', 'weights', 'seed'] in __init__() have been deprecated since v4.4.0. They will be replaced with the corresponding keyword arguments in v6.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v4.4.0 for details.
  study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\samplers\_tpe\sampler.py:82: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  optuna_warn(msg, FutureW

Best Validation Accuracy: 0.8924
Optimal Hyperparameters:
  - unfreeze_pct: 0.4
  - learning_rate: 2.068638831086326e-05
  - optimizer: adamw


In [21]:
# Re-train model with best hyperparameters
best_params = study.best_params
final_model = build_mobilenet_v2().to(device)
final_model.load_state_dict(warm_state_dict)

# Apply best unfreeze depth
for param in final_model.features.parameters():
    param.requires_grad = False
        
total_blocks = len(list(final_model.features.children()))
num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
for block in list(final_model.features.children())[-num_unfreeze_blocks:]:
    for param in block.parameters():
        param.requires_grad = True

# Apply best optimizer
trainable_params = [p for p in final_model.parameters() if p.requires_grad]
if best_params["optimizer"] == "adamw":
    final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
else:
    final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

# Train final model
for epoch in range(PHASE_2_EPOCHS):
    train_loss, train_acc = train_one_epoch(final_model, train_loader, criterion, final_opt)
    val_loss, val_acc = validate(final_model, val_loader, criterion)
    print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

Phase 2 - Epoch 6/15 | Train Acc: 0.8058 | Val Acc: 0.8088
Phase 2 - Epoch 7/15 | Train Acc: 0.8366 | Val Acc: 0.8127
Phase 2 - Epoch 8/15 | Train Acc: 0.8519 | Val Acc: 0.8327
Phase 2 - Epoch 9/15 | Train Acc: 0.8767 | Val Acc: 0.8566
Phase 2 - Epoch 10/15 | Train Acc: 0.8935 | Val Acc: 0.8367
Phase 2 - Epoch 11/15 | Train Acc: 0.8960 | Val Acc: 0.8645
Phase 2 - Epoch 12/15 | Train Acc: 0.9054 | Val Acc: 0.8685
Phase 2 - Epoch 13/15 | Train Acc: 0.9044 | Val Acc: 0.8685
Phase 2 - Epoch 14/15 | Train Acc: 0.9128 | Val Acc: 0.8566
Phase 2 - Epoch 15/15 | Train Acc: 0.9307 | Val Acc: 0.8805


In [22]:
# Export model to Open Neural Network Exchange (ONNX) to be run outside PyTorch
# Export to ONNX
final_model.eval()  # Set model in evaluation mode
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    final_model,
    dummy_input,
    MobileNetV2_ONNX_EXPORT_PATH,
    export_params=True,
    opset_version=14,   # Version of ONNX
    do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
    input_names=["input"],  # Key to be referred when giving input during inference
    output_names=["output"],    # Key to be referred when obtaining output during inference
    dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

# Delete model
del final_model

In [23]:
# Test model
# Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
# Load session with CUDA acceleration
session = ort.InferenceSession(
    MobileNetV2_ONNX_EXPORT_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"Active Provider  : {session.get_providers()[0]}")

# GPU warmup (so that latency is accurate)
dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

for _ in range(20):
    session.run([output_name], {input_name: dummy_warmup_input})

# Synchronize GPU after warmup
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Test accuracy and latency
total_samples = 0
correct_predictions = 0
latency_list = []

for images, labels in test_loader:
    # Convert PyTorch tensor to NumPy array (Float32)
    input_numpy = images.numpy()
    
    # Precise GPU Timing Block
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Clear pending GPU tasks
        
    start_time = time.perf_counter()
    
    # Run Inference
    outputs = session.run([output_name], {input_name: input_numpy})
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Wait for GPU to finish computing
        
    end_time = time.perf_counter()
    
    # Calculate Latency in milliseconds
    batch_latency_ms = (end_time - start_time) * 1000
    latency_list.append(batch_latency_ms)
    
    # Accuracy Calculation
    logits = outputs[0]
    preds = np.argmax(logits, axis=1)
    correct_predictions += np.sum(preds == labels.numpy())
    total_samples += images.size(0)

# Display results
avg_accuracy = (correct_predictions / total_samples)
avg_latency = np.mean(latency_list)

print(f"Test Accuracy: {avg_accuracy:.4f}")
print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8755
Average Inference Latency: 1.944 ms / sample


In [24]:
# Run remaining 4 seeds for MobileNetV2
test_accuracies = []
latencies_ms = []
SEEDS = [0, 198, 473, 5784, 96312]

test_accuracies.append(avg_accuracy)
latencies_ms.append(avg_latency)

for seed in SEEDS:
    # Skip the first seed
    if seed == 0:
        continue

    # Set seed
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True    
    torch.backends.cudnn.benchmark = False
    print(f'\nSeed: {seed}')

    # Fine tune MobileNetV2 model
    # Phase 1
    model = build_mobilenet_v2().to(device)
    
    # Freeze feature backbone
    for param in model.features.parameters():
        param.requires_grad = False

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

    for epoch in range(PHASE_1_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Phase 2
    # Apply best unfreeze depth
    for param in model.features.parameters():
        param.requires_grad = False
        
    total_blocks = len(list(model.features.children()))
    num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
    for block in list(model.features.children())[-num_unfreeze_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    # Apply best optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if best_params["optimizer"] == "adamw":
        final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
    else:
        final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

    # Train final model
    for epoch in range(PHASE_2_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, final_opt)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    MobileNetV2_ONNX_EXPORT_PATH_adjust_with_seed = f'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/MobileNetV2-Seed-{seed}.onnx'

    # Export to ONNX
    model.eval()  # Set model in evaluation mode
    dummy_input = torch.randn(1, 3, 224, 224, device=device)

    torch.onnx.export(
        model,
        dummy_input,
        MobileNetV2_ONNX_EXPORT_PATH_adjust_with_seed,
        export_params=True,
        opset_version=14,   # Version of ONNX
        do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
        input_names=["input"],  # Key to be referred when giving input during inference
        output_names=["output"],    # Key to be referred when obtaining output during inference
        dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
            "input": {0: "batch_size"},
            "output": {0: "batch_size"}
        }
    )

    # Delete model
    del model

    # Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
    # Load session with CUDA acceleration
    session = ort.InferenceSession(
        MobileNetV3_Small_ONNX_EXPORT_PATH_adjust_with_seed,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    print(f"Active Provider  : {session.get_providers()[0]}")

    # GPU warmup (so that latency is accurate)
    dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

    for _ in range(20):
        session.run([output_name], {input_name: dummy_warmup_input})

    # Synchronize GPU after warmup
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Test accuracy and latency
    total_samples = 0
    correct_predictions = 0
    latency_list = []

    for images, labels in test_loader:
        # Convert PyTorch tensor to NumPy array (Float32)
        input_numpy = images.numpy()
    
        # Precise GPU Timing Block
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Clear pending GPU tasks
        
        start_time = time.perf_counter()
    
        # Run Inference
        outputs = session.run([output_name], {input_name: input_numpy})
    
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Wait for GPU to finish computing
        
        end_time = time.perf_counter()
    
        # Calculate Latency in milliseconds
        batch_latency_ms = (end_time - start_time) * 1000
        latency_list.append(batch_latency_ms)
    
        # Accuracy Calculation
        logits = outputs[0]
        preds = np.argmax(logits, axis=1)
        correct_predictions += np.sum(preds == labels.numpy())
        total_samples += images.size(0)

    # Display results
    avg_accuracy = (correct_predictions / total_samples)
    avg_latency = np.mean(latency_list)

    print(f"Test Accuracy: {avg_accuracy:.4f}")
    print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

    test_accuracies.append(avg_accuracy)
    latencies_ms.append(avg_latency)


Seed: 198
Phase 1 - Epoch 1/5 | Train Acc: 0.5260 | Val Acc: 0.6853
Phase 1 - Epoch 2/5 | Train Acc: 0.7058 | Val Acc: 0.7371
Phase 1 - Epoch 3/5 | Train Acc: 0.7246 | Val Acc: 0.7331
Phase 1 - Epoch 4/5 | Train Acc: 0.7647 | Val Acc: 0.7530
Phase 1 - Epoch 5/5 | Train Acc: 0.7528 | Val Acc: 0.7729
Phase 2 - Epoch 6/15 | Train Acc: 0.7821 | Val Acc: 0.8008
Phase 2 - Epoch 7/15 | Train Acc: 0.8227 | Val Acc: 0.8367
Phase 2 - Epoch 8/15 | Train Acc: 0.8717 | Val Acc: 0.8167
Phase 2 - Epoch 9/15 | Train Acc: 0.8648 | Val Acc: 0.8446
Phase 2 - Epoch 10/15 | Train Acc: 0.8821 | Val Acc: 0.8207
Phase 2 - Epoch 11/15 | Train Acc: 0.8920 | Val Acc: 0.8526
Phase 2 - Epoch 12/15 | Train Acc: 0.9009 | Val Acc: 0.8685
Phase 2 - Epoch 13/15 | Train Acc: 0.9183 | Val Acc: 0.8845
Phase 2 - Epoch 14/15 | Train Acc: 0.9108 | Val Acc: 0.8765
Phase 2 - Epoch 15/15 | Train Acc: 0.9252 | Val Acc: 0.8845
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8677
Average Inference Latency: 1.872 ms / sam

In [25]:
# Record results
MobileNetV2_results = pd.DataFrame({
    'seed': SEEDS,
    'test accuracy': test_accuracies,
    'latency in millisecond': latencies_ms
})

MobileNetV2_results.to_csv('MobileNetV2-results.csv', index = False)

In [32]:
def build_efficientnet_b0(num_classes=NUM_CLASSES):
    # Load pre-trained EfficientNet-Lite0
    model = models.efficientnet_b0(pretrained=True)
    
    # Extract input features from the existing linear head
    in_features = model.classifier[1].in_features
    
    # Replace the final classification head
    model._fc = nn.Sequential(
        nn.Dropout(p=0.3),
        nn.Linear(in_features, num_classes)
    )
    
    return model

In [36]:
# Set seed
random.seed(0)
np.random.seed(0)
torch.backends.cudnn.deterministic = True    
torch.backends.cudnn.benchmark = False

# Set batch size
BATCH_SIZE = 32

# Set save model path
EfficientNetB0_ONNX_EXPORT_PATH = 'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/EfficientNetB0.onnx'

In [37]:
# Fine tune MobileNetV2 model
model = build_efficientnet_b0().to(device)
    
# Freeze feature backbone
for param in model.features.parameters():
    param.requires_grad = False

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

for epoch in range(PHASE_1_EPOCHS):
    train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
    val_loss, val_acc = validate(model, val_loader, criterion)
    print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

# Save warm weights in memory for Optuna trials
warm_state_dict = copy.deepcopy(model.state_dict())

# Remove model
del model

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Phase 1 - Epoch 1/5 | Train Acc: 0.2843 | Val Acc: 0.4821
Phase 1 - Epoch 2/5 | Train Acc: 0.5939 | Val Acc: 0.6534
Phase 1 - Epoch 3/5 | Train Acc: 0.6751 | Val Acc: 0.6773
Phase 1 - Epoch 4/5 | Train Acc: 0.7117 | Val Acc: 0.7211
Phase 1 - Epoch 5/5 | Train Acc: 0.7474 | Val Acc: 0.7450


In [38]:
# Instantiate Optuna Study (Hyperparameter tuning using Bayesian optimisation)
study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
objective_fn = OptunaObjective(warm_state_dict, train_loader, val_loader, build_efficientnet_b0)
    
study.optimize(objective_fn, n_trials=OPTUNA_TRIALS)

# Output Best Results
print(f"Best Validation Accuracy: {study.best_value:.4f}")
print("Optimal Hyperparameters:")
for key, value in study.best_params.items():
    print(f"  - {key}: {value}")

C:\Users\user\AppData\Local\Temp\ipykernel_28924\2316673851.py:2: FutureWarning: __init__() got {'consider_prior'} as positional arguments but they were expected to be given as keyword arguments.
Positional arguments ['self', 'consider_prior', 'prior_weight', 'consider_magic_clip', 'consider_endpoints', 'n_startup_trials', 'n_ei_candidates', 'gamma', 'weights', 'seed'] in __init__() have been deprecated since v4.4.0. They will be replaced with the corresponding keyword arguments in v6.0.0, so please use the keyword specification instead. See https://github.com/optuna/optuna/releases/tag/v4.4.0 for details.
  study = optuna.create_study(direction="maximize", sampler=optuna.samplers.TPESampler(0))
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\optuna\samplers\_tpe\sampler.py:82: FutureWarning: `consider_prior` has been deprecated in v4.3.0. This feature will be removed in v6.0.0. See https://github.com/optuna/optuna/releases/tag/v4.3.0.
  optuna_warn(msg, FutureW

Best Validation Accuracy: 0.8725
Optimal Hyperparameters:
  - unfreeze_pct: 0.1
  - learning_rate: 3.130998793121403e-05
  - optimizer: adamw


In [39]:
# Re-train model with best hyperparameters
best_params = study.best_params
final_model = build_efficientnet_b0().to(device)
final_model.load_state_dict(warm_state_dict)

# Apply best unfreeze depth
for param in final_model.features.parameters():
    param.requires_grad = False
        
total_blocks = len(list(final_model.features.children()))
num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
for block in list(final_model.features.children())[-num_unfreeze_blocks:]:
    for param in block.parameters():
        param.requires_grad = True

# Apply best optimizer
trainable_params = [p for p in final_model.parameters() if p.requires_grad]
if best_params["optimizer"] == "adamw":
    final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
else:
    final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

# Train final model
for epoch in range(PHASE_2_EPOCHS):
    train_loss, train_acc = train_one_epoch(final_model, train_loader, criterion, final_opt)
    val_loss, val_acc = validate(final_model, val_loader, criterion)
    print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\torchvision\models\_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=EfficientNet_B0_Weights.IMAGENET1K_V1`. You can also use `weights=EfficientNet_B0_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


Phase 2 - Epoch 6/15 | Train Acc: 0.7771 | Val Acc: 0.7729
Phase 2 - Epoch 7/15 | Train Acc: 0.8063 | Val Acc: 0.7849
Phase 2 - Epoch 8/15 | Train Acc: 0.8351 | Val Acc: 0.8127
Phase 2 - Epoch 9/15 | Train Acc: 0.8588 | Val Acc: 0.8207
Phase 2 - Epoch 10/15 | Train Acc: 0.8826 | Val Acc: 0.8327
Phase 2 - Epoch 11/15 | Train Acc: 0.8915 | Val Acc: 0.8247
Phase 2 - Epoch 12/15 | Train Acc: 0.9034 | Val Acc: 0.8566
Phase 2 - Epoch 13/15 | Train Acc: 0.9148 | Val Acc: 0.8486
Phase 2 - Epoch 14/15 | Train Acc: 0.9227 | Val Acc: 0.8486
Phase 2 - Epoch 15/15 | Train Acc: 0.9371 | Val Acc: 0.8645


In [40]:
# Export model to Open Neural Network Exchange (ONNX) to be run outside PyTorch
# Export to ONNX
final_model.eval()  # Set model in evaluation mode
dummy_input = torch.randn(1, 3, 224, 224, device=device)

torch.onnx.export(
    final_model,
    dummy_input,
    EfficientNetB0_ONNX_EXPORT_PATH,
    export_params=True,
    opset_version=14,   # Version of ONNX
    do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
    input_names=["input"],  # Key to be referred when giving input during inference
    output_names=["output"],    # Key to be referred when obtaining output during inference
    dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
        "input": {0: "batch_size"},
        "output": {0: "batch_size"}
    }
)

# Delete model
del final_model

In [41]:
# Test model
# Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
# Load session with CUDA acceleration
session = ort.InferenceSession(
    EfficientNetB0_ONNX_EXPORT_PATH,
    providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
)

input_name = session.get_inputs()[0].name
output_name = session.get_outputs()[0].name

print(f"Active Provider  : {session.get_providers()[0]}")

# GPU warmup (so that latency is accurate)
dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

for _ in range(20):
    session.run([output_name], {input_name: dummy_warmup_input})

# Synchronize GPU after warmup
if torch.cuda.is_available():
    torch.cuda.synchronize()

# Test accuracy and latency
total_samples = 0
correct_predictions = 0
latency_list = []

for images, labels in test_loader:
    # Convert PyTorch tensor to NumPy array (Float32)
    input_numpy = images.numpy()
    
    # Precise GPU Timing Block
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Clear pending GPU tasks
        
    start_time = time.perf_counter()
    
    # Run Inference
    outputs = session.run([output_name], {input_name: input_numpy})
    
    if torch.cuda.is_available():
        torch.cuda.synchronize()  # Wait for GPU to finish computing
        
    end_time = time.perf_counter()
    
    # Calculate Latency in milliseconds
    batch_latency_ms = (end_time - start_time) * 1000
    latency_list.append(batch_latency_ms)
    
    # Accuracy Calculation
    logits = outputs[0]
    preds = np.argmax(logits, axis=1)
    correct_predictions += np.sum(preds == labels.numpy())
    total_samples += images.size(0)

# Display results
avg_accuracy = (correct_predictions / total_samples)
avg_latency = np.mean(latency_list)

print(f"Test Accuracy: {avg_accuracy:.4f}")
print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\onnxruntime\capi\onnxruntime_inference_collection.py:153: UserWarning: Specified provider 'CUDAExecutionProvider' is not in available provider names.Available providers: 'AzureExecutionProvider, CPUExecutionProvider'
  warnings.warn(


Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8599
Average Inference Latency: 4.892 ms / sample


In [42]:
# Run remaining 4 seeds for MobileNetV2
test_accuracies = []
latencies_ms = []
SEEDS = [0, 198, 473, 5784, 96312]

test_accuracies.append(avg_accuracy)
latencies_ms.append(avg_latency)

for seed in SEEDS:
    # Skip the first seed
    if seed == 0:
        continue

    # Set seed
    random.seed(seed)
    np.random.seed(seed)
    torch.backends.cudnn.deterministic = True    
    torch.backends.cudnn.benchmark = False
    print(f'\nSeed: {seed}')

    # Fine tune EfficientNetB0 model
    # Phase 1
    model = build_efficientnet_b0().to(device)
    
    # Freeze feature backbone
    for param in model.features.parameters():
        param.requires_grad = False

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.classifier.parameters(), lr=1e-3)

    for epoch in range(PHASE_1_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, optimizer)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 1 - Epoch {epoch+1}/{PHASE_1_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    # Phase 2
    # Apply best unfreeze depth
    for param in model.features.parameters():
        param.requires_grad = False
        
    total_blocks = len(list(model.features.children()))
    num_unfreeze_blocks = int(total_blocks * best_params["unfreeze_pct"])
    for block in list(model.features.children())[-num_unfreeze_blocks:]:
        for param in block.parameters():
            param.requires_grad = True

    # Apply best optimizer
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    if best_params["optimizer"] == "adamw":
        final_opt = optim.AdamW(trainable_params, lr=best_params["learning_rate"], weight_decay=1e-4)
    else:
        final_opt = optim.SGD(trainable_params, lr=best_params["learning_rate"], momentum=0.9)

    # Train final model
    for epoch in range(PHASE_2_EPOCHS):
        train_loss, train_acc = train_one_epoch(model, train_loader, criterion, final_opt)
        val_loss, val_acc = validate(model, val_loader, criterion)
        print(f"Phase 2 - Epoch {PHASE_1_EPOCHS+epoch+1}/{PHASE_1_EPOCHS+PHASE_2_EPOCHS} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

    EfficientNetB0_ONNX_EXPORT_PATH_adjust_with_seed = f'C:/Users/user/Documents/Master of Computer Science/Flagship project/Garbage Classification/trained models/EfficientNet-B0-Seed-{seed}.onnx'

    # Export to ONNX
    model.eval()  # Set model in evaluation mode
    dummy_input = torch.randn(1, 3, 224, 224, device=device)

    torch.onnx.export(
        model,
        dummy_input,
        EfficientNetB0_ONNX_EXPORT_PATH_adjust_with_seed,
        export_params=True,
        opset_version=14,   # Version of ONNX
        do_constant_folding=True,   # Pre-compute some static scale factor (reduce size and inference time)
        input_names=["input"],  # Key to be referred when giving input during inference
        output_names=["output"],    # Key to be referred when obtaining output during inference
        dynamic_axes={  # Specify that input and output batch size is dynamic (can be 0 or 16 or 32)
            "input": {0: "batch_size"},
            "output": {0: "batch_size"}
        }
    )

    # Delete model
    del model

    # Preload PyTorch CUDA DLLs to ensure onnxruntime finds the GPU on Windows
    # Load session with CUDA acceleration
    session = ort.InferenceSession(
        MobileNetV3_Small_ONNX_EXPORT_PATH_adjust_with_seed,
        providers=['CUDAExecutionProvider', 'CPUExecutionProvider']
    )

    input_name = session.get_inputs()[0].name
    output_name = session.get_outputs()[0].name

    print(f"Active Provider  : {session.get_providers()[0]}")

    # GPU warmup (so that latency is accurate)
    dummy_warmup_input = np.random.randn(1, 3, 224, 224).astype(np.float32)

    for _ in range(20):
        session.run([output_name], {input_name: dummy_warmup_input})

    # Synchronize GPU after warmup
    if torch.cuda.is_available():
        torch.cuda.synchronize()

    # Test accuracy and latency
    total_samples = 0
    correct_predictions = 0
    latency_list = []

    for images, labels in test_loader:
        # Convert PyTorch tensor to NumPy array (Float32)
        input_numpy = images.numpy()
    
        # Precise GPU Timing Block
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Clear pending GPU tasks
        
        start_time = time.perf_counter()
    
        # Run Inference
        outputs = session.run([output_name], {input_name: input_numpy})
    
        if torch.cuda.is_available():
            torch.cuda.synchronize()  # Wait for GPU to finish computing
        
        end_time = time.perf_counter()
    
        # Calculate Latency in milliseconds
        batch_latency_ms = (end_time - start_time) * 1000
        latency_list.append(batch_latency_ms)
    
        # Accuracy Calculation
        logits = outputs[0]
        preds = np.argmax(logits, axis=1)
        correct_predictions += np.sum(preds == labels.numpy())
        total_samples += images.size(0)

    # Display results
    avg_accuracy = (correct_predictions / total_samples)
    avg_latency = np.mean(latency_list)

    print(f"Test Accuracy: {avg_accuracy:.4f}")
    print(f"Average Inference Latency: {avg_latency:.3f} ms / sample")

    test_accuracies.append(avg_accuracy)
    latencies_ms.append(avg_latency)


Seed: 198
Phase 1 - Epoch 1/5 | Train Acc: 0.2838 | Val Acc: 0.4980
Phase 1 - Epoch 2/5 | Train Acc: 0.5820 | Val Acc: 0.6414
Phase 1 - Epoch 3/5 | Train Acc: 0.6736 | Val Acc: 0.6932
Phase 1 - Epoch 4/5 | Train Acc: 0.7112 | Val Acc: 0.7131
Phase 1 - Epoch 5/5 | Train Acc: 0.7444 | Val Acc: 0.7331
Phase 2 - Epoch 6/15 | Train Acc: 0.7702 | Val Acc: 0.7649
Phase 2 - Epoch 7/15 | Train Acc: 0.8153 | Val Acc: 0.7968
Phase 2 - Epoch 8/15 | Train Acc: 0.8276 | Val Acc: 0.7928
Phase 2 - Epoch 9/15 | Train Acc: 0.8588 | Val Acc: 0.8088
Phase 2 - Epoch 10/15 | Train Acc: 0.8856 | Val Acc: 0.8207
Phase 2 - Epoch 11/15 | Train Acc: 0.8831 | Val Acc: 0.8406
Phase 2 - Epoch 12/15 | Train Acc: 0.9074 | Val Acc: 0.8486
Phase 2 - Epoch 13/15 | Train Acc: 0.9128 | Val Acc: 0.8406
Phase 2 - Epoch 14/15 | Train Acc: 0.9237 | Val Acc: 0.8526
Phase 2 - Epoch 15/15 | Train Acc: 0.9321 | Val Acc: 0.8685
Active Provider  : CPUExecutionProvider
Test Accuracy: 0.8677
Average Inference Latency: 2.135 ms / sam

In [43]:
# Record results
EfficientNetB0_results = pd.DataFrame({
    'seed': SEEDS,
    'test accuracy': test_accuracies,
    'latency in millisecond': latencies_ms
})

EfficientNetB0_results.to_csv('EfficientNet-B0-results.csv', index = False)